In [ ]:
pip install gensim

In [ ]:
from gensim.models import Word2Vec
import numpy as np
import re



In [ ]:
# ----------------------
# Dataset
# ----------------------

# Download latest version
import pandas as pd
import os
import kagglehub

path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
df = pd.read_csv(os.path.join(path, "IMDB Dataset.csv"))
df = df[["review", "sentiment"]].dropna()
df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})
sentences = list(zip(df["review"], df["label"]))

# Show what loaded
print(df.head())
print(df.info())

import random

random.seed(42)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train_sentences = sentences[:split]
test_sentences = sentences[split:]

sentences = train_sentences

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
                                              review sentiment  label
0  One of the other reviewers has mentioned that ...  positive      1
1  A wonderful little production. <br /><br />The...  positive      1
2  I thought this was a wonderful way to spend ti...  positive      1
3  Basically there's a family where a little boy ...  negative      0
4  Petter Mattei's "Love in the Time of Money" is...  positive      1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
 2   label      50000 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.1+ MB
None


In [ ]:
# ----------------------
# Preprocessing
# ----------------------
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

In [ ]:
corpus = [tokenize(text) for text, _ in sentences]
print(corpus)

In [ ]:
# ----------------------
# Train Word2Vec
# ----------------------
model = Word2Vec(corpus, vector_size=50, window=4, min_count=1, sg=1)

In [ ]:
# ----------------------
# Sentence vector
# ----------------------
def sentence_vector(tokens):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

In [ ]:
X = np.array([sentence_vector(tokens) for tokens in corpus])
y = np.array([label for _, label in sentences])

In [ ]:
# ----------------------
# Simple sentiment prototypes
# ----------------------
pos_vec = np.mean(X[y == 1], axis=0)
neg_vec = np.mean(X[y == 0], axis=0)

In [ ]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def predict(sentence):
    v = sentence_vector(tokenize(sentence))
    return "positive" if cosine(v, pos_vec) > cosine(v, neg_vec) else "negative"

In [ ]:
# ----------------------
# Try it
# ----------------------
sentences = test_sentences

label_map = {1: "positive", 0: "negative"}

correct = 0
for text, true_label in sentences:
    pred = predict(text)

    # Convert the 1 or 0 from your data into "positive" or "negative"
    if pred == label_map[true_label]:
        correct += 1

accuracy = correct / len(sentences)
print(f"Actual Accuracy: {accuracy * 100}%")

tests = [
    "great movie",
    "painfully slow and boring",
    "not bad but not great",
    "I loved the visuals"
]

for t in tests:
    print(t, "→", predict(t))

Actual Accuracy: 72.19%
great movie → positive
painfully slow and boring → negative
not bad but not great → negative
I loved the visuals → positive
